In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import time

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

SEED = 42
DEVICE = torch.device("cpu")
torch.manual_seed(SEED)

print("Device:", DEVICE)

Device: cpu


In [6]:
# ============================================================
# CELL 1: IMPORTS AND CLASSROOM CONFIGURATION
# ============================================================
# Purpose:
# Import the libraries required to build our RAG system.
#
# LIBRARIES:
#
# os, Path, dotenv:
#   Read environment variables and locate the .env file.
#
# time:
#   Measure how long model inference takes.
#
# re, json:
#   Work with text patterns and JSON data.
#
# pandas:
#   Display documents and numerical vectors as tables.
#
# numpy:
#   Perform mathematical operations on numerical vectors.
#
# matplotlib:
#   Visualize vectors and mathematical relationships.
#
# Ollama:
#   Connect to our locally running language model.
#
# CountVectorizer:
#   Convert text into numerical word-count vectors.
#
# TfidfVectorizer:
#   Convert text into numerical TF-IDF vectors.
#
# cosine_similarity:
#   Calculate similarity between document and query vectors.
#
# SentenceTransformer:
#   Load a pretrained neural embedding model for semantic search.
#
# Hugging Face:
#   Authenticate and download pretrained embedding models.
# ============================================================

# Standard Python libraries.
import os
import re
import time
import json
from pathlib import Path

# Environment variables and authentication.
from dotenv import load_dotenv
from huggingface_hub import HfApi

# Data processing and mathematical calculations.
import pandas as pd
import numpy as np

# Visualization.
import matplotlib.pyplot as plt

# Local language model inference.
# from ollama import Client

# Text vectorization.
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Vector similarity.
from sklearn.metrics.pairwise import cosine_similarity

# Neural embedding models.
from sentence_transformers import SentenceTransformer


# ============================================================
# CLASSROOM CONFIGURATION
# ============================================================

MODEL_NAME = "qwen2.5:0.5b"
OLLAMA_HOST = "http://localhost:11434"

TOP_K = 2
TEMPERATURE = 0
MAX_OUTPUT_TOKENS = 100

# Connect to the local Ollama server.
# client = Client(host=OLLAMA_HOST)

print("Model:", MODEL_NAME)
print("Retrieval Method: TF-IDF")
print("Top-K:", TOP_K)

Model: qwen2.5:0.5b
Retrieval Method: TF-IDF
Top-K: 2


In [8]:
# ============================================================
# CELL 9A: BUILD A TRANSFORMER BLOCK
# ============================================================
# Purpose:
# A Transformer block combines:
#
# 1. Multi-head self-attention
# 2. Residual connections
# 3. Layer normalization
# 4. Feed-forward neural network
#
# Multi-head attention allows the model to learn different
# attention patterns simultaneously.
#
# Residual connections help information and gradients flow
# through the network.
#
# Layer normalization stabilizes representations.
#
# The feed-forward network transforms each token's features.
#
# PyTorch provides MultiheadAttention, which implements
# the Q/K/V projections and attention computation.
#
# We use batch_first=True:
# (batch, sequence, embedding_dimension)
# ============================================================

class TransformerBlock(nn.Module):

    def __init__(self, embedding_dim, num_heads, feedforward_dim):
        super().__init__()

        # Main Component 1
        self.attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=num_heads,
            dropout=0.0,
            batch_first=True)

        # Supporting Component 1
        self.ln1 = nn.LayerNorm(embedding_dim)
        self.ln2 = nn.LayerNorm(embedding_dim)

        # Main Component 2
        self.feedforward = nn.Sequential(
            nn.Linear(embedding_dim, feedforward_dim),
            nn.ReLU(),
            nn.Linear(feedforward_dim, embedding_dim))


    def forward(self, x):

        T = x.shape[1]


        # True means a position is blocked from attention.
        mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)

        # Supporting Component 2
        # Attention + residual connection.
        normalized = self.ln1(x)
        attended, _ = self.attention(normalized, normalized, normalized, attn_mask=mask, need_weights=False)

        # Residual 1: Original representation + attention output
        x = x + attended

        # Residual 2: Updated representation + feed-forward output
        # Feed-forward network + residual connection.
        x = x + self.feedforward(self.ln2(x))

        return x

In [9]:
# ============================================================
# CELL 9B: BUILDING OUR COMPLETE TINY TRANSFORMER
# ============================================================
# Purpose:
# Combine all the components into one language model.
#
# Architecture:
#
# Token IDs
#    ↓
# Token Embeddings + Position Embeddings
#    ↓
# Transformer Block(s)
#    ↓
# Layer Normalization
#    ↓
# Linear Output Layer
#    ↓
# Logits for every vocabulary token
#
# The output tensor has shape:
#
# (batch_size, sequence_length, vocabulary_size)
#
# Each position produces scores for the NEXT character.
# ============================================================

class TinyTransformerLM(nn.Module):

    def __init__(self, vocab_size, embedding_dim, num_heads, num_layers, feedforward_dim, context_length):
        super().__init__()

        self.context_length = context_length

        self.token_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.position_embedding = nn.Embedding(context_length, embedding_dim)

        self.blocks = nn.Sequential(*[
            TransformerBlock(embedding_dim, num_heads, feedforward_dim)
            for _ in range(num_layers)
        ])

        self.ln_final = nn.LayerNorm(embedding_dim)
        self.output = nn.Linear(embedding_dim, vocab_size)

    def forward(self, input_ids):

        B, T = input_ids.shape

        if T > self.context_length:
            raise ValueError("Input sequence exceeds context length.")

        positions = torch.arange(T, device=input_ids.device)

        x = self.token_embedding(input_ids) + self.position_embedding(positions)
        x = self.blocks(x)
        x = self.ln_final(x)

        return self.output(x)

In [10]:
# ============================================================
# CELL 12A: DEFINE TEXT GENERATION (BEFORE TRAINING)
# ============================================================
# Purpose:
# This function works before AND after training.
#
# Now we perform INFERENCE.
#
# Inference means using learned parameters to produce
# predictions without updating the model.
#
# Generation works autoregressively:
#
# 1. Read the current sequence.
# 2. Predict logits for the next character.
# 3. Convert logits into probabilities.
# 4. Sample a character.
# 5. Append it to the sequence.
# 6. Repeat.
#
# max_new_tokens controls how many new characters we generate.
#
# Because our tokenizer is character-level:
#
# ONE TOKEN = ONE CHARACTER.
# ============================================================

@torch.no_grad()
def generate_text(model, prompt, max_new_tokens=100, temperature=1.0):
    """
    Generate text autoregressively using our untrained Transformer.

    Parameters
    ----------
    model : TinyTransformerLM
        Untrained language model.
    prompt : str
        Starting text. Characters must exist in the vocabulary.
    max_new_tokens : int
        Number of new characters to generate.
    temperature : float
        Sampling temperature. Must be greater than zero.

    Returns
    -------
    str
        Original prompt plus generated continuation.
    """

    if temperature <= 0:
        raise ValueError("Temperature must be greater than zero.")

    was_training = model.training
    model.eval()

    ids = torch.tensor([encode(prompt)], dtype=torch.long, device=DEVICE)

    for _ in range(max_new_tokens):

        # Keep only the most recent context_length characters.
        context = ids[:, -model.context_length:]

        # Predict logits for every position.
        logits = model(context)

        # Only the final position predicts the next character.
        next_logits = logits[:, -1, :] / temperature

        # Convert logits into probabilities.
        probabilities = F.softmax(next_logits, dim=-1)

        # Sample one character.
        next_id = torch.multinomial(probabilities, num_samples=1)

        # Append the generated character.
        ids = torch.cat([ids, next_id], dim=1)

    if was_training:
        model.train()
    return decode(ids[0].tolist())

In [12]:

checkpoint = torch.load(
    "a7_experiment_4.pt",
    map_location=DEVICE,
    weights_only=True
)

config = checkpoint["config"]
CONTEXT_LENGTH = config["context_length"]

# Restore the original character-level vocabulary.
chars = checkpoint["vocab"]
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

def encode(text):
    return [stoi[ch] for ch in text]

def decode(ids):
    return "".join(itos[int(i)] for i in ids)

# Recreate the architecture from Experiment 4.
selected_model = TinyTransformerLM(
    vocab_size=len(chars),
    embedding_dim=config["embedding_dim"],
    num_heads=config["num_heads"],
    num_layers=config["num_layers"],
    feedforward_dim=config["feedforward_dim"],
    context_length=config["context_length"]
).to(DEVICE)

selected_model.load_state_dict(checkpoint["model_state"])
selected_model.eval()

print("Selected SLM loaded successfully.")
print("Configuration:", config)
print("Parameters:", sum(
    p.numel() for p in selected_model.parameters()
))


Selected SLM loaded successfully.
Configuration: {'embedding_dim': 48, 'num_heads': 2, 'num_layers': 2, 'feedforward_dim': 192, 'context_length': 32}
Parameters: 60795


In [16]:

# A7 PART 2 - STEP 2.1: KNOWLEDGE BASE

documents = [
    {
        "id": "D1",
        "title": "hours",
        "text": "CC cafe opens at seven."
    },
    {
        "id": "D2",
        "title": "drinks",
        "text": "CC cafe serves tea."
    },
    {
        "id": "D3",
        "title": "food",
        "text": "CC cafe sells soup."
    },
    {
        "id": "D4",
        "title": "location",
        "text": "CC cafe is in urbana."
    },
    {
        "id": "D5",
        "title": "delivery",
        "text": "CC cafe offers delivery."
    }
]

display(pd.DataFrame(documents))


,id,title,text
0,D1,hours,CC cafe opens at seven.
1,D2,drinks,CC cafe serves tea.
2,D3,food,CC cafe sells soup.
3,D4,location,CC cafe is in urbana.
4,D5,delivery,CC cafe offers delivery.


In [17]:

questions = [
    {
        "question": "when does CC cafe open?",
        "expected": "seven"
    },
    {
        "question": "what does CC cafe serve?",
        "expected": "tea"
    },
    {
        "question": "what food does CC cafe sell?",
        "expected": "soup"
    },
    {
        "question": "where is CC cafe?",
        "expected": "urbana"
    },
    {
        "question": "does CC cafe offer delivery?",
        "expected": "yes"
    }
]

# Wrapper for the original SLM generation function.

def ask_slm(prompt, temperature=0.7, max_new_tokens=60):
    torch.manual_seed(SEED)
    selected_model.eval()

    # Convert to lowercase and remove unsupported characters.
    prompt = prompt.lower()
    prompt = "".join(ch for ch in prompt if ch in stoi)

    generated = generate_text(
        selected_model,
        prompt,
        max_new_tokens=max_new_tokens,
        temperature=temperature
    )

    return generated[len(prompt):]


baseline_results = []

for item in questions:
    response = ask_slm(item["question"])

    baseline_results.append({
        "Question": item["question"],
        "Expected Answer": item["expected"],
        "SLM Response": response
    })

baseline_df = pd.DataFrame(baseline_results)
display(baseline_df)


,Question,Expected Answer,SLM Response
0,when does CC cafe open?,seven,prodataining parameters.\nthe professor teach...
1,what does CC cafe serve?,tea,nive startup applications.\ndjango application...
2,what food does CC cafe sell?,soup,language models can run on local computers.\n...
3,where is CC cafe?,urbana,rtup applications.\ndjango applications use mo...
4,does CC cafe offer delivery?,yes,offers courses in informatics.\nstudents expe...


## Part 2 — Step 2.2: Baseline SLM Evaluation (Without RAG)

We evaluated our selected SLM from Part 1 (Experiment 4) using five factual questions about CC Cafe, without providing any retrieved documents or additional context.

The model did not produce relevant answers to the five questions. Instead, it generated text resembling the original training corpus, including references to programming, Django, artificial intelligence, and university courses.

This behavior is expected because the model was trained on a small, unrelated dataset and has not learned the specific facts about CC Cafe.

These baseline responses will be compared with the RAG-generated responses to determine whether providing external knowledge improves factual accuracy and relevance.

In [18]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2)
)

document_matrix = vectorizer.fit_transform([
    doc["title"] + " " + doc["text"]
    for doc in documents
])

def retrieve(query, top_k=1):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(
        query_vector, document_matrix
    )[0]

    ranked_indices = scores.argsort()[::-1][:top_k]

    return [
        {
            **documents[i],
            "score": float(scores[i])
        }
        for i in ranked_indices
    ]

print(retrieve("where is CC Cafe?", top_k=2))

[{'id': 'D4', 'title': 'location', 'text': 'CC cafe is in urbana.', 'score': 0.38146185225535234}, {'id': 'D2', 'title': 'drinks', 'text': 'CC cafe serves tea.', 'score': 0.3193023297639811}]


In [19]:
# STEP 2.3: BUILD RAG PROMPT

def build_rag_prompt(question, matches):
    context = " ".join(
        doc["text"] for doc in matches
    )

    return question + "\n" + context + "\n"

In [20]:

# STEP 2.3: RUN 20 RAG EXPERIMENTS

top_k_values = [1, 2, 3, 5]
rag_results = []

for item in questions:
    question = item["question"]

    for k in top_k_values:
        matches = retrieve(question, top_k=k)
        prompt = build_rag_prompt(question, matches)

        response = ask_slm(prompt)

        rag_results.append({
            "Question": question,
            "Top-K": k,
            "Expected Answer": item["expected"],
            "Retrieved IDs": ", ".join(
                doc["id"] for doc in matches
            ),
            "Retrieved Documents": " | ".join(
                doc["text"] for doc in matches
            ),
            "Generated Response": response,
            "Quality Score (1-10)": None
        })

rag_df = pd.DataFrame(rag_results)
display(rag_df)

print("Total RAG experiments:", len(rag_df))

rag_df.to_csv("a7_rag_results.csv", index=False)


,Question,Top-K,Expected Answer,Retrieved IDs,Retrieved Documents,Generated Response,Quality Score (1-10)
0,when does CC cafe open?,1,seven,D4,CC cafe is in urbana.,machine learning models contain parameters.\nt...,None
1,when does CC cafe open?,2,seven,"D4, D2",CC cafe is in urbana. | CC cafe serves tea.,machine learning models contain parameters.\nt...,None
2,when does CC cafe open?,3,seven,"D4, D2, D3",CC cafe is in urbana. | CC cafe serves tea. | ...,smartificial intelligence systems.\nlanguage m...,None
3,when does CC cafe open?,5,seven,"D4, D2, D3, D1, D5",CC cafe is in urbana. | CC cafe serves tea. | ...,the models and views.\ntraining changes the we...,None
4,what does CC cafe serve?,1,tea,D4,CC cafe is in urbana.,machine learning models contain parameters.\nt...,None
5,what does CC cafe serve?,2,tea,"D4, D2",CC cafe is in urbana. | CC cafe serves tea.,machine learning models contain parameters.\nt...,None
6,what does CC cafe serve?,3,tea,"D4, D2, D3",CC cafe is in urbana. | CC cafe serves tea. | ...,smartificial intelligence systems.\nlanguage m...,None
7,what does CC cafe serve?,5,tea,"D4, D2, D3, D1, D5",CC cafe is in urbana. | CC cafe serves tea. | ...,the models and views.\ntraining changes the we...,None
8,what food does CC cafe sell?,1,soup,D3,CC cafe sells soup.,smartificial intelligence systems.\nlanguage m...,None
9,what food does CC cafe sell?,2,soup,"D3, D4",CC cafe sells soup. | CC cafe is in urbana.,machine learning models contain parameters.\nt...,None


Total RAG experiments: 20


In [21]:

# Print every generated answer without truncation.

for i, row in rag_df.iterrows():
    print(f"\nExperiment {i + 1}")
    print("Question:", row["Question"])
    print("Top-K:", row["Top-K"])
    print("Expected:", row["Expected Answer"])
    print("Retrieved:", row["Retrieved IDs"])
    print("Generated:", repr(row["Generated Response"]))
    print("-" * 60)



Experiment 1
Question: when does CC cafe open?
Top-K: 1
Expected: seven
Retrieved: D4
Generated: 'machine learning models contain parameters.\nthe professor te'
------------------------------------------------------------

Experiment 2
Question: when does CC cafe open?
Top-K: 2
Expected: seven
Retrieved: D4, D2
Generated: 'machine learning models contain parameters.\nthe professor te'
------------------------------------------------------------

Experiment 3
Question: when does CC cafe open?
Top-K: 3
Expected: seven
Retrieved: D4, D2, D3
Generated: 'smartificial intelligence systems.\nlanguage models predict t'
------------------------------------------------------------

Experiment 4
Question: when does CC cafe open?
Top-K: 5
Expected: seven
Retrieved: D4, D2, D3, D1, D5
Generated: 'the models and views.\ntraining changes the weights of a neur'
------------------------------------------------------------

Experiment 5
Question: what does CC cafe serve?
Top-K: 1
Expected: tea
Retrieve

In [22]:
# Use only if all 20 responses are incorrect and irrelevant.
rag_df["Quality Score (1-10)"] = [1] * 20

display(rag_df)
rag_df.to_csv("a7_rag_results.csv", index=False)

,Question,Top-K,Expected Answer,Retrieved IDs,Retrieved Documents,Generated Response,Quality Score (1-10)
0,when does CC cafe open?,1,seven,D4,CC cafe is in urbana.,machine learning models contain parameters.\nt...,1
1,when does CC cafe open?,2,seven,"D4, D2",CC cafe is in urbana. | CC cafe serves tea.,machine learning models contain parameters.\nt...,1
2,when does CC cafe open?,3,seven,"D4, D2, D3",CC cafe is in urbana. | CC cafe serves tea. | ...,smartificial intelligence systems.\nlanguage m...,1
3,when does CC cafe open?,5,seven,"D4, D2, D3, D1, D5",CC cafe is in urbana. | CC cafe serves tea. | ...,the models and views.\ntraining changes the we...,1
4,what does CC cafe serve?,1,tea,D4,CC cafe is in urbana.,machine learning models contain parameters.\nt...,1
5,what does CC cafe serve?,2,tea,"D4, D2",CC cafe is in urbana. | CC cafe serves tea.,machine learning models contain parameters.\nt...,1
6,what does CC cafe serve?,3,tea,"D4, D2, D3",CC cafe is in urbana. | CC cafe serves tea. | ...,smartificial intelligence systems.\nlanguage m...,1
7,what does CC cafe serve?,5,tea,"D4, D2, D3, D1, D5",CC cafe is in urbana. | CC cafe serves tea. | ...,the models and views.\ntraining changes the we...,1
8,what food does CC cafe sell?,1,soup,D3,CC cafe sells soup.,smartificial intelligence systems.\nlanguage m...,1
9,what food does CC cafe sell?,2,soup,"D3, D4",CC cafe sells soup. | CC cafe is in urbana.,machine learning models contain parameters.\nt...,1


## Part 2 — RAG Experiment Results

### Experimental Setup

We evaluated our selected Small Language Model from Part 1 (Experiment 4) using five factual questions about CC Cafe.

Our knowledge base contained five documents covering opening hours, drinks, food, location, and delivery.

We used TF-IDF and cosine similarity to retrieve documents with top-K values of 1, 2, 3, and 5, resulting in 20 RAG experiments.

The model weights remained unchanged throughout the RAG experiments.

### Retrieval Results

The retrieval system successfully identified the correct document at top-K=1 for three of the five questions: food, location, and delivery.

However, the opening-hours and drinks questions retrieved the location document first. This demonstrates that TF-IDF retrieval does not always identify the most relevant document, especially when several documents contain overlapping terms.

Increasing K eventually included the correct documents for all questions, but retrieving more documents did not guarantee better answers.

### Generation Results

The SLM primarily generated text resembling its original training corpus instead of directly answering questions about CC Cafe.

This occurred even when the retrieved documents contained the correct information.

One major limitation was the model's 32-character context window. Long RAG prompts exceeded this window, preventing the model from accessing much of the earlier information.

Furthermore, the small character-level SLM was trained for next-character prediction rather than factual question answering or instruction following.

We evaluated each response on factual accuracy, relevance to the question, and support from retrieved documents. All 20 responses received 1/10 because none provided the requested CC Cafe fact, and the generated text was unrelated to the retrieved evidence. These scores reflect generation quality rather than retrieval accuracy.

### Overall Observation

The RAG pipeline successfully retrieved relevant information in several experiments, but the SLM generally failed to use that information to produce accurate factual answers.

These results demonstrate that retrieval quality and generation quality are separate components of a RAG system. Providing correct information does not guarantee a correct response, particularly when the generator has limited capacity and context length.

## Part 2 — Analysis Questions

**1. How did RAG change the model's responses compared with the baseline?**

RAG changed some of the generated text, but the responses remained largely unrelated to the CC Cafe questions. The model continued generating sentences resembling its original training corpus instead of using the retrieved facts.

**2. Did RAG improve factual accuracy? Why or why not?**

RAG did not noticeably improve factual accuracy in our experiments. Although the retrieval system sometimes returned the correct information, the SLM struggled to use it because it had a small context window and was not trained specifically for factual question answering.

**3. What happened when more documents were retrieved?**

Increasing K included more potentially relevant information, but it did not consistently improve generated responses. Additional documents increased the length of the prompt, and some information was pushed outside the model's 32-character context window.

**4. How did the limited context window affect the results?**

The SLM could only attend to the most recent 32 characters during generation. Consequently, longer prompts could cause the model to lose access to the original question and earlier retrieved documents. This limited its ability to use multiple pieces of retrieved evidence.

**5. How does this small character-level SLM compare with a larger language model for RAG?**

Our SLM has only 60,795 parameters and uses character-level tokenization. Unlike a larger instruction-tuned language model, it has limited capacity to interpret questions, follow instructions, and integrate external knowledge. As a result, the retrieval pipeline could find useful documents, but the SLM often failed to generate answers supported by them.